# 🎓 WE4 · Notebook 03 — Actor–Critic
## Hiring someone to tell you what a learner is *worth*

> **This notebook picks up exactly where notebook 02 stopped.** Same company, same campaign, same
> code — so before anything new, here is the state of play in five lines.
>
> You run retention at **Owlinguo**, a free language-learning app that earns **one ad before every
> lesson**. When a learner's streak breaks you have **three days** to win them back, and each morning
> you choose one move: leave them alone, send a nudge from their coach, or blast them with ads. The
> retention team wants **one sheet of paper with three lines** — *learner looks like this → do that*
> — and in notebook 02 you learned that sheet from experience rather than from opinion, using
> **REINFORCE**.
>
> The rule you learned it with, in plain words: **run a batch of campaigns, and for each move you
> made, push its probability up or down depending on whether that campaign went better or worse than
> a campaign usually goes from that situation.**

That rule works, and it found the best sheet. But the way we measured *"better or worse than usual"*
was improvised in two places, and both improvisations cost us something real. This notebook names
what those two quantities actually were, and the algorithm reshapes itself around the answer.

**What we are going to do**
1. Take the two terms we improvised and replace them with the objects they were imitating: the
   **Q-function**, the **value function**, and the **advantage**.
2. Discover that once they are named properly, we no longer need to wait for a campaign to end —
   a single day, `(s, a, r, s′)`, is enough to learn from.
3. Notice we don't *know* those functions — and do the obvious thing an ML person does with an
   unknown function: **learn it**. That learner is the **critic**.
4. Put the two together into **A2C**, the advantage actor–critic — the shape that essentially every
   modern RL system, including the ones that fine-tune language models, is built on.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- **Interactive widgets** to play with each idea *before* the maths shows up.
- Same tiny campaign as before — small enough that we can compute **every quantity exactly** and
  check our learned estimates against the truth. Use that hard; you will never have it again.
- 💰 All money is expected ad revenue per learner, in **CHF**.

> 🧭 **Nothing here assumes you memorised notebook 02.** Part 1 re-shows the campaign, hands back the
> functions you wrote there, and re-derives the one formula we build on. Run its cells: they define
> the world everything else uses.

## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `ac_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 🔑 **While the course repo is private** (testing phase) you need a GitHub access token:
> open the **Secrets** panel (🔑 icon in the left sidebar), add a secret named
> **`GITHUB_TOKEN`**, paste your token, and toggle *Notebook access* on. Once the repo is
> public, no token is needed — the cell clones it directly.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE4-public"
HELPER     = os.path.join("3_actor_critic", "exercise", "ac_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding `3_actor_critic/exercise/` — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (3_actor_critic/exercise/ac_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "3_actor_critic", "exercise"))
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r 3_actor_critic/exercise/requirements_ac.txt

**0.3 — Import the libraries.** The diagrams, widgets and quizzes live in **`ac_viz`** so the
teaching cells stay about the *idea*, not about HTML.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import itertools

import importlib
import ac_viz
importlib.reload(ac_viz)   # pick up the latest helpers even if a stale copy was cached

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

print("Environment ready ✅  ·  torch", torch.__version__)
print("GPU visible to this runtime:", torch.cuda.is_available())
print("…and we will not use it. Our critic is a 2-layer network with a 6-number input: shipping it")
print("to a GPU would cost more in transfer than it saves in arithmetic. The whole notebook trains")
print("in a few seconds on the CPU. Knowing when NOT to reach for the accelerator is part of the job.")

---
# Part 1 — Where notebook 02 left us

Same app, same three-day campaign, same numbers. Nothing about the *world* changes today — what
changes is how we learn from it.

In [ ]:
ac_viz.campaign_recap()

### The world, and the functions you already wrote

These are yours from notebook 02, handed back so we can get to the new material. Run the cell and
skim it: `transition` rolls the world's dice, `returns_to_go` scores each day by its own future, and
`action_probs` is the softmax policy over each state's row of logits.

In [ ]:
ENGAGE, ACTIONS   = ac_viz.ENGAGE, ac_viz.ACTIONS
TRANS, REWARD     = ac_viz.TRANS, ac_viz.REWARD
START_PROBS       = ac_viz.START_PROBS
N_DAYS            = 3
GAMMA             = 0.9        # Owlinguo values tomorrow's franc at 90% of today's

def transition(state, action):
    '''The environment's one-step rule: returns (next state, reward).'''
    reward   = REWARD[state][action]
    outcomes = TRANS[state][action]                    # [(next state, probability), ...]
    next_state = np.random.choice([s for s, p in outcomes], p=[p for s, p in outcomes])
    return int(next_state), float(reward)

def returns_to_go(rewards, gamma=GAMMA):
    '''G_t = r_t + gamma*r_{t+1} + ...  for every t.'''
    out = [0.0] * len(rewards)
    for t in reversed(range(len(rewards))):
        out[t] = rewards[t] + gamma * (out[t + 1] if t + 1 < len(rewards) else 0.0)
    return out

def action_probs(theta, state):
    '''π(·|s) — the softmax of that state's row of logits.'''
    return torch.softmax(theta[state], dim=-1)

print("A 🙂 Warm learner who gets a 🔔 Nudge:", transition(1, 1), " (next state, reward booked)")
print("Returns-to-go of the campaign [-0.5, 0.5, 6.0]:", np.round(returns_to_go([-0.5, 0.5, 6.0]), 3))

### The learning rule we finished with

Over notebook 02 that rule got two rounds of noise reduction. First, each move was scored by **its
own future** rather than by the whole campaign's takings — money booked *before* a move cannot be
that move's doing. Then, that score was compared against **what a campaign normally collects from
that engagement level**, measured as the average over the batch.

Put together, this was the best-behaved version of the score gradient we reached — and it is the one
line this entire notebook is a critique of:

In [ ]:
ac_viz.score_gradient_recap()

### The two things it left unfinished

Both were flagged in passing as we went; neither was fixed. They are the whole agenda of today.

In [ ]:
ac_viz.where_we_left_off()

### 🎯 Task 1 — one small addition to the episode sampler

This is notebook 02's `sample_episode`, with **one new list**: `next_states`. Until now we only ever
needed the states we *visited*; from Part 3 onwards every update will be built out of a complete
four-tuple **(s, a, r, s′)** — where we were, what we did, what it paid, and **where the learner ended
up**. So we record `s′` as it happens.

Two blanks, both recall from notebook 02.

In [ ]:
def sample_episode(theta):
    '''One 3-day campaign. Returns states, actions, rewards, NEXT states, and the
    log-probabilities of the actions we took.'''
    engagement = int(np.random.choice(len(ENGAGE), p=START_PROBS))     # which learner walked in today
    states, actions, rewards, next_states, log_probs = [], [], [], [], []
    for day in range(N_DAYS):
        p = action_probs(theta, engagement)
        action = int(torch.multinomial(p, 1))
        log_probs.append(torch.log(p[???]))          # 🎯 the log-prob of the action we ACTUALLY took
        next_engagement, reward = transition(engagement, action)
        states.append(engagement); actions.append(action); rewards.append(reward)
        next_states.append(???)                      # 🎯 the s′ of this step — where did the learner land?
        engagement = next_engagement
    return states, actions, rewards, next_states, log_probs

theta = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)     # uniform policy, no opinions yet
s, a, r, s_next, lp = sample_episode(theta)
print("states     :", [ENGAGE[x] for x in s])
print("next states:", [ENGAGE[x] for x in s_next])
assert s_next[:-1] == s[1:], "Today's s′ IS tomorrow's s — the lists should overlap by one day."
print("\n✅ Today's s′ is tomorrow's s. The only new thing is the LAST one: where the learner was")
print("   left on day 3, after the campaign closed.")

---
# Part 2 — Naming the two terms we improvised

Here is the bracket we have to deal with, one term at a time:

$$\underbrace{G_t}_{\text{term 1}} \;-\; \underbrace{b(s_t)}_{\text{term 2}}$$

## 2.1 · Term 1: the return-to-go is a *sample* of something

Write out what `G_t` actually is for one campaign:

$$G_t \;=\; r_t \;+\; \gamma r_{t+1} \;+\; \gamma^2 r_{t+2} \;+\;\dots$$

Every one of those rewards happened **after** dice were rolled — ours (which action did the policy
sample tomorrow?) and the world's (did the nudge land?). Run the same campaign again from the same
morning with the same first move and you get a **different** `G_t`.

In other words, `G_t` describes **one campaign that happened**, not the move that started it. It is
**one draw** from all the futures that move could have led to. The number those draws are scattered
around — the one that really does belong to the pair `(s, a)` — is something you have already met in
the lecture: the **state–action value function**, `Q`.

$$\boxed{\;Q^\pi(s, a) \;=\; \mathbb{E}_{\tau \sim \pi}\Big[\, G_t \;\Big|\; s_t = s,\; a_t = a \,\Big]\;}$$

The subscript is the important part of that line: the average is taken over **the futures our own
policy π produces** (together with the world's dice). Change how we behave from tomorrow on, and this
number changes.

> **In words:** *"Take action `a` in state `s` right now, then carry on behaving the way I normally
> do. On average, what does the rest of the campaign pay?"*

Two things to notice, and they matter later:
- the **π superscript**. Q is not a fact about the world — it depends on how we behave afterwards.
- it commits to `a` **only for one step**. After that, it is business as usual.

### Let's see that scatter

We fix the policy, and we fix one situation and one move. Then we let the campaign play out from
there, over and over.

In [ ]:
# A policy with some opinions — this is "how we normally behave" for the whole of Part 2.
theta_pi = torch.tensor([[0.0,  0.8, -0.5],      # 😴 Cold  → mostly 🔔 Nudge
                         [0.0,  0.8, -0.5],      # 🙂 Warm  → mostly 🔔 Nudge
                         [0.0, -0.5,  0.8]])     # 🔥 Hot   → mostly 📺 Ad blast

ac_viz.fixed_setup(theta_pi, day=0, state=1, action=1)

Now replay the rest of that campaign **4000 times** and collect the `G` each replay comes back
with. (The exact `Q` and `V` drawn on top are computed in §2.2 — for now, just look at where the
samples pile up.)

In [ ]:
ac_viz.g_samples_q(theta_pi, day=0, state=1, action=1, gamma=GAMMA, n=4000)

**That histogram is the whole problem with notebook 02.** Every single campaign handed our
gradient one bar from it — a number that could land anywhere between 0 and 9 — when the quantity we
actually wanted was the red line. We were not *wrong*; we were **loud**.

So let us fix that one term, and only that one. Set the baseline aside for a moment (it comes back in
§2.2, with a proper name of its own) and look at the rule as it was **before** we subtracted
anything — each move weighted by the return-to-go it happened to collect:

$$\nabla_\theta J \;=\; \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_t \gamma^t\,
{\color{#c0554e}G_t}\,\nabla_\theta \log \pi_\theta(a_t\mid s_t)\Big]$$

`G_t` is a noisy sample of `Q^π(s_t, a_t)`, so put the thing itself in instead of the sample:

$$\nabla_\theta J \;=\; \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_t \gamma^t\,
{\color{#4a5bd0}Q^\pi(s_t,a_t)}\,\nabla_\theta \log \pi_\theta(a_t\mid s_t)\Big]$$

Same expectation, one term replaced. This is a theorem (the *policy gradient theorem*, in its Q form)
and we will take it as given — the proof is a page of algebra that adds nothing to the intuition. The
intuition is the histogram: **replacing a sample by the number it is scattered around cannot change
an average, but it removes all of that spread.**

### 🧠 Quick check — what is a return-to-go?

In [ ]:
ac_viz.mc_quiz("g_is_q")

## 2.2 · Term 2: the baseline we *wanted* also has a name

Go back to what we said a baseline should be, in notebook 02:

> *"An outcome is only good or bad relative to **what you normally expect in that situation**."*

Write that sentence as a formula — *the expected return-to-go from state `s`, behaving as we
normally do* — and you have written down the other function from the lecture, the **(state) value
function**:

$$\boxed{\;V^\pi(s) \;=\; \mathbb{E}_{\tau \sim \pi}\Big[\, G_t \;\Big|\; s_t = s \,\Big]
\;=\; \sum_a \pi(a\mid s)\, Q^\pi(s,a)\;}$$

**V is Q with the action averaged out.** Q asks *"what if I do this?"*; V asks *"what is this
situation worth to me at all?"* — before any particular move is picked, weighting each move by how
often we would actually pick it.

> ⚠️ **V is not "the best we could do here".** It is what *this* policy gets here, mistakes
> included — a promising situation handled by a poor policy has a low V. That is not a defect, it is
> exactly why V is the right yardstick: we want to know whether a move beats **our current habit**,
> not whether it beats perfection. Nobody knows what perfection looks like; everybody can measure
> their own habit.

### 🎯 Task 2 — compute V and Q exactly (the toy-world luxury)

Our campaign is three days long and has three states, so we can compute both functions **exactly**,
by walking the campaign **backwards** from its last morning:

$$Q^\pi(t, s, a) \;=\; \underbrace{R(s,a)}_{\text{booked today}} \;+\;
\gamma \underbrace{\sum_{s'} P(s'\mid s,a)\, V^\pi(t+1, s')}_{\text{what tomorrow is worth, on average}}
\qquad\qquad
V^\pi(t, s) \;=\; \sum_a \pi(a\mid s)\, Q^\pi(t,s,a)$$

Note the extra index `t`: in a campaign that *ends*, a 🔥 Hot learner on day 0 is worth much more than
a 🔥 Hot learner on the last morning, simply because more campaign remains. We start the recursion
from **`V(3, ·) = 0`** — after day 3 the campaign is closed and there is nothing left to collect.

Two reminders about the tables, so you can write this without hunting:
> 💡 **`REWARD[s][a]`** — indexed **state first, action second** — is the money that move books today.
> And `TRANS[s][a]` is the list of `(next state, probability)` pairs, which is why
> `sum(p * V[t+1][ns] for ns, p in TRANS[s][a])` is the weighted average of tomorrow's values. That
> one is already written for you as `tomorrow`.

In [ ]:
def exact_values(theta, gamma=GAMMA):
    '''V and Q of the policy `theta`, computed exactly by walking the campaign backwards.
    Returns V with shape (3 days, 3 states) and Q with shape (3 days, 3 states, 3 actions).'''
    V = np.zeros((N_DAYS + 1, len(ENGAGE)))                    # V[3] stays 0: the campaign is over
    Q = np.zeros((N_DAYS, len(ENGAGE), len(ACTIONS)))

    for t in reversed(range(N_DAYS)):                          # last morning first
        for s in range(len(ENGAGE)):
            pi = action_probs(theta, s).detach().numpy()       # π(·|s): how we normally behave here
            for a in range(len(ACTIONS)):
                tomorrow = sum(p * V[t + 1][ns] for ns, p in TRANS[s][a])
                Q[t, s, a] = ???        # 🎯 the money this move books today, plus `tomorrow` discounted by gamma
            #                             🎯 V is Q averaged over the actions, weighted by how often π picks each
            V[t, s] = sum(??? for a in range(len(ACTIONS)))
    return V[:N_DAYS], Q

V_PI, Q_PI = exact_values(theta_pi)

assert abs(Q_PI[2, 2, 2] - REWARD[2][2]) < 1e-9, "On the last morning Q is just today's reward."
assert abs(V_PI[0, 1] - 3.731) < 1e-2 and abs(Q_PI[0, 1, 1] - 5.090) < 1e-2, \
    "Those are the two numbers the histogram above was scattered around."
print("✅ V and Q agree with the histogram. The 4000 campaigns were sampling THIS number:")
print("   Q(day 0, 🙂 Warm, 🔔 Nudge) =", round(float(Q_PI[0, 1, 1]), 3))

In [ ]:
ac_viz.value_table(V_PI)

> 👀 **Two things worth reading off that table.** Down each column, value **falls** as the days
> pass — the campaign is running out, and a learner is worth what remains of it. And 😴 Cold on the
> last morning is worth *less than zero*: the only thing our policy does with a Cold learner is spend
> money nudging them, and on day 2 there is no tomorrow left for that investment to pay off in.

**One more thing that table gives us for free.** The objective from notebook 02 — the expected return
of the whole campaign — is now a one-liner: it is just the value of the learners who walk in.

$$J(\theta) \;=\; \sum_{s} \rho(s)\, V^\pi(0, s)$$

We will use this as our exact scoreboard for the rest of the notebook, instead of enumerating
trajectories.

In [ ]:
def exact_J(theta, gamma=GAMMA):
    '''The exact objective: the average value of a learner walking into the campaign.'''
    V, _ = exact_values(theta, gamma)
    return float(np.dot(START_PROBS, V[0]))

def sheet_J(sheet, gamma=GAMMA):
    '''Exact J of a hand-written deterministic sheet (one action per engagement level).'''
    logits = torch.full((len(ENGAGE), len(ACTIONS)), -20.0)
    for s, a in enumerate(sheet):
        logits[s, a] = 20.0
    return exact_J(logits, gamma)

J_UNIFORM = exact_J(torch.zeros(len(ENGAGE), len(ACTIONS)))
J_BEST    = max(sheet_J(list(sh)) for sh in itertools.product(range(len(ACTIONS)), repeat=len(ENGAGE)))

print(f"J(uniform policy)   = {J_UNIFORM:+.3f}")
print(f"J(theta_pi)         = {exact_J(theta_pi):+.3f}   ← the opinionated policy we're studying")
print(f"J(best of 27 sheets) = {J_BEST:+.3f}   ← the target to beat, as in notebook 02")

### 🧠 Quick check — V and Q

In [ ]:
ac_viz.true_false_quiz("values")

## 2.3 · Put them together: the **advantage**

Both terms of that bracket now have proper names, so let's substitute them in:

$$\underbrace{G_t}_{\text{a sample of } Q^\pi(s,a)} \;-\; \underbrace{b(s_t)}_{\text{a stand-in for } V^\pi(s)}
\qquad\longrightarrow\qquad
\boxed{\;A^\pi(s,a) \;=\; Q^\pi(s,a) \;-\; V^\pi(s)\;}$$

This is the **advantage function**, and it answers the only question the actor needs answered:

> ### *"Is this move better or worse than what I would normally do in this situation?"*

Not *"is this a good state"* — that is V's job, and it is none of the actor's business. Not *"is this
the best possible action"* — we have no idea what the best possible action is. Just: **better or
worse than my own habit, right here.**

Play with the three functions together. Pick a morning and a learner and watch how the same three Q
values turn into three advantages once the V line is subtracted.

In [ ]:
ac_viz.q_v_a_explorer(Q_PI, V_PI)

In [ ]:
ac_viz.advantage_meaning()

### 🔢 Work three of them out by hand

The reward table is reprinted inside the quiz, so everything you need is on screen.

In [ ]:
ac_viz.number_quiz("vqa")

### 🧠 Quick check — the advantage

In [ ]:
ac_viz.true_false_quiz("advantage")

## 2.4 · The theorem that changes the shape of the algorithm

Substituting the advantage into the policy gradient gives the form everything from here on uses:

$$\nabla_\theta J(\theta) \;=\; \mathbb{E}_{\tau\sim\pi_\theta}
\Big[\sum_{t} \gamma^{t}\, A^\pi(s_t,a_t)\, \nabla_\theta \log \pi_\theta(a_t\mid s_t)\Big]$$

And now the step that matters. That sum over `t` inside an expectation over whole trajectories can be
rewritten as a **single expectation over (state, action) pairs**, where the pairs are drawn from
however often the policy actually visits them:

$$\boxed{\;\nabla_\theta J(\theta) \;\propto\; \mathbb{E}_{(s,a)\,\sim\,\pi_\theta}
\Big[\,A^\pi(s,a)\; \nabla_\theta \log \pi_\theta(a\mid s)\,\Big]\;}$$

Take a second on what disappeared: **the time index, and the trajectory**. There is no `t` and no `τ`
left. Nothing in that expression knows which day of the campaign it came from, or which campaign, or
whether the campaign has finished.

**So a single step is now a complete, legitimate training example.** Where notebook 02 needed a whole
finished campaign to score even its first day, this form needs one `(s, a)` pair and its advantage.
That is **Problem 1 solved** — as soon as we can get our hands on `A`.

We take this equivalence as given too; the proof is bookkeeping about visitation distributions.

> 📎 **A small honesty note about `γ^t`.** Strictly, the `γ^t` factor is absorbed into how often each
> `(s,a)` is counted. In practice essentially every implementation — ours included — drops it and
> weights every step equally. It slightly re-weights early days versus late ones, everyone does it,
> and nobody has ever regretted it.

---
# Part 3 — One day is enough

We have a beautiful formula with an unusable ingredient: `A^π(s,a) = Q^π(s,a) − V^π(s)` needs **two**
functions we do not have. Before worrying about how to get them, notice that we only need **one**.

Look again at the definition of Q, and read it as a sentence:

$$Q^\pi(s,a) \;=\; \underbrace{R(s,a)}_{\text{the money this move books today}} \;+\;
\gamma\,\underbrace{\mathbb{E}_{s' \sim P(\cdot\mid s,a)}\big[V^\pi(s')\big]}_{\text{what the learner is worth tomorrow}}$$

*"What this move is worth = what it pays now + what it leaves me holding."* And when we **observe** a
transition, the world hands us a sample of both pieces: `r` is the money, `s′` is what we are left
holding. So for one observed step:

$$Q^\pi(s,a) \;\approx\; r + \gamma V^\pi(s')
\qquad\Longrightarrow\qquad
\boxed{\;A^\pi(s,a) \;\approx\; \underbrace{r + \gamma V^\pi(s') - V^\pi(s)}_{\textbf{the TD error, } \delta}\;}$$

**One unknown function left.** That is why a critic learns `V` and never bothers with `Q`.

In [ ]:
ac_viz.one_step_diagram()

### 🎯 Task 3 — the one-step advantage

Write that box in code. It takes a value table `V` and one observed transition, and returns the
advantage estimate for the move we made.

One wrinkle to handle: `V(s′)` on the **last day**. After day 3 the campaign is closed and the
learner is worth nothing more to it, so the value of any state at day 3 is **0**. The helper
`value_of` below encodes exactly that convention — use it for *both* lookups so you never index off
the end of the table.

In [ ]:
def value_of(V, day, state):
    '''V(day, state), with the convention that a finished campaign is worth 0.'''
    return 0.0 if day >= N_DAYS else float(V[day][state])

def td_advantage(V, day, state, reward, next_state, gamma=GAMMA):
    '''A(s,a) estimated from ONE observed transition: r + γ·V(s′) − V(s).'''
    target = ???          # 🎯 today's money, plus what tomorrow's learner is worth (discounted)
    return target - ???   # 🎯 minus what we expected of this situation before we picked anything

# --- self-check 1: the last morning. Nothing follows, so the advantage is just today's reward
#     compared to what the policy usually earns there.
a_last = td_advantage(V_PI, 2, 2, REWARD[2][2], 0)      # 🔥 Hot, 📺 Ad blast, on day 2
assert abs(a_last - (REWARD[2][2] - V_PI[2, 2])) < 1e-9, "On day 2 there is no tomorrow to add."
print(f"A(day 2, 🔥 Hot, 📺 Ad blast) = {a_last:+.3f}   (vs the exact {Q_PI[2,2,2] - V_PI[2,2]:+.3f})")

# --- self-check 2: average many one-step estimates of the SAME (s,a) and watch them find the truth
draws = []
for _ in range(4000):
    s_next, reward = transition(1, 1)                   # 🙂 Warm + 🔔 Nudge, over and over
    draws.append(td_advantage(V_PI, 0, 1, reward, s_next))
print(f"\nmean of 4000 one-step estimates : {np.mean(draws):+.3f}")
print(f"the exact A(day 0, 🙂 Warm, 🔔 Nudge): {Q_PI[0,1,1] - V_PI[0,1]:+.3f}  ✅")

### One campaign, judged as it happens

Here is what that buys us, concretely. Run one campaign and score **each morning on the morning
after**, using nothing but that day's four-tuple.

In [ ]:
np.random.seed(7)
states, actions, rewards, next_states, _ = sample_episode(theta_pi)

steps = [{"day": t, "s": states[t], "a": actions[t], "r": rewards[t], "s_next": next_states[t],
          "v": value_of(V_PI, t, states[t]), "v_next": value_of(V_PI, t + 1, next_states[t]),
          "delta": td_advantage(V_PI, t, states[t], rewards[t], next_states[t])}
         for t in range(N_DAYS)]
ac_viz.online_update_demo(steps)

> 🔭 **We did change something, and it is worth naming.** The weight on an action used to be
> *the whole rest of the campaign, as it actually happened*; it is now *today's reward, plus a
> **prediction** of the rest*. That swap is the classic trade of the field — far less noise, at the
> price of inheriting whatever the prediction gets wrong. **Notebook 04 is largely about that trade**:
> **GAE** gives you a dial that sits anywhere between the two, and you will measure both ends of it
> there. For today, take the one-step version and keep moving.

---
# Part 4 — We don't know V. So we learn it.

Everything above assumed a `V` we do not have — we only had one because our world is a toy with a
published transition table. On a real problem there is no `exact_values()`.

But look at what we are missing: **a function**. Give it a situation, it returns a number. And in
machine learning, an unknown function you can collect examples of is not a wall — it is a
**regression problem**. So: a small neural network, trained on experience.

The only question is what to train it *against*. There is no dataset of true values to copy.

## 4.1 · TD learning: the target you build out of your own guess

The lecture gave us the answer, and Part 3 already wrote it down without saying so. Rearrange the
definition of V one step forward:

$$V^\pi(s) \;=\; \mathbb{E}_{a \sim \pi(\cdot\mid s),\; s' \sim P(\cdot\mid s,a)}\big[\, r + \gamma V^\pi(s')\,\big]$$

*"What this situation is worth = what today pays + what tomorrow's situation is worth."* That is a
**consistency condition** — and a consistency condition is something you can train towards, even when
you know neither side. Observe a transition, and the right-hand side becomes a number you can
compute:

$$\underbrace{r + \gamma V(s')}_{\textbf{TD target}} \qquad\text{versus}\qquad
\underbrace{V(s)}_{\text{what we currently say}}
\qquad\Longrightarrow\qquad
\delta = \text{target} - V(s) \quad\textbf{(the TD error)}$$

and the update is the one every incremental estimator uses — *move a little towards what you just
saw*:

$$V(s) \;\leftarrow\; V(s) + \alpha\,\delta$$

There is something slightly unnerving about this, and it is worth naming: **the target contains our
own estimate**. We are learning a guess from a guess. It is called **bootstrapping**, and the reason
it works is that one end of the target is *real* — the observed reward `r` anchors it. Reality leaks
in one day at a time, and drags the whole function into place.

Watch it happen. One estimate, one observation at a time, α on a slider.

In [ ]:
ac_viz.td_playground()

### 🔢 One TD update, by hand

In [ ]:
ac_viz.number_quiz("td")

### 🧠 Quick check — TD learning

In [ ]:
ac_viz.true_false_quiz("td")

## 4.2 · The critic, as a network

Two design decisions before the code, both worth a sentence.

**What does the critic look at?** The engagement level **and the day**. Remember the policy is *not*
allowed to see the day — the retention team wants one rule per engagement level. The critic is under
no such constraint: it never picks an action, so it cannot bias the update. This is the same
permission that let us consider a `b(day, engagement)` baseline in notebook 02, now used properly.

**How does that get into a network?** As a **one-hot** vector: 3 slots for the day, 3 for the
engagement level, exactly one `1` in each half. Six numbers in, one number out.

> 💡 With six possible inputs, a lookup table would honestly do. We use a network because the point
> is the *shape* of the solution: on a real problem the state is a customer embedding or a game frame,
> and then the network is the only option. Nothing else in this notebook changes.

In [ ]:
class Critic(nn.Module):
    '''V̂(day, engagement) — a two-layer network. One number in the output: what this
    situation is worth for the rest of the campaign.'''
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(N_DAYS + len(ENGAGE), hidden), nn.Tanh(),
                                 nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)      # (batch, 1) → (batch,)

def features(days, states):
    '''(day, engagement) → the 6-number one-hot input the critic reads.'''
    days, states = torch.as_tensor(days), torch.as_tensor(states)
    return torch.cat([F.one_hot(days.clamp(max=N_DAYS - 1), N_DAYS),
                      F.one_hot(states, len(ENGAGE))], dim=-1).float()

def V_hat(critic, days, states):
    '''The critic's V — with the same convention as `value_of`: a finished campaign is worth 0.'''
    days = torch.as_tensor(days)
    v = critic(features(days, states))
    return torch.where(days >= N_DAYS, torch.zeros_like(v), v)

# --- meet it on one example before we use it in anger
torch.manual_seed(0)
demo_critic = Critic()
print("one-hot input for (day 1, 🙂 Warm):", features([1], [1]).numpy()[0])
print("V̂(day 1, 🙂 Warm)  =", round(float(V_hat(demo_critic, [1], [1])), 3), " ← untrained: noise")
print("V̂(day 3, 🙂 Warm)  =", round(float(V_hat(demo_critic, [3], [1])), 3), " ← the campaign is over")

### 🎯 Task 4 — the critic's target and the critic's loss

Two short functions. Both work on **whole batches** of transitions at once (tensors), which is why
there are no loops — but the formula is exactly the scalar one from §4.1.

**(a) The TD target.** `r + γ·V̂(s′)`. The lookup of `V̂(s′)` is already written for you — note it
asks the critic about `day + 1`, because `s′` is tomorrow's learner.

**(b) The loss.** The critic is doing plain regression: predict `V̂(s)`, aim at the target, penalise
the square of the gap. `L = mean( (V̂(s) − target)² )`.

> 💡 In torch, `x ** 2` squares every entry of a tensor, and `.mean()` averages a tensor down to a
> single number — so a mean squared error is one short expression, no loop.

> ⚠️ **The one subtlety, and it matters.** The target is built from the critic's *own* output, so
> without care the gradient would flow into it too — and the network could cheat by dragging the
> target towards its prediction instead of the other way round. `torch.no_grad()` in the scaffolding
> below freezes the target into a plain number: **the target is a label, not a variable.**

In [ ]:
def td_target(critic, days, rewards, next_states, gamma=GAMMA):
    '''The TD target for a batch of transitions — a constant, as far as autograd is concerned.'''
    with torch.no_grad():                       # ← the target is a label; do not differentiate it
        next_value = V_hat(critic, days + 1, next_states)   # what the critic thinks tomorrow is worth
        return ???      # 🎯 today's rewards, plus that value discounted by gamma

def critic_loss(v_pred, target):
    '''Plain regression: how far is the critic's prediction from the target it should have hit?'''
    return ???          # 🎯 the mean of the squared gap between the two

# --- self-check on a handful of hand-made transitions
d  = torch.tensor([0, 1, 2])                       # three steps, one per day
st = torch.tensor([1, 2, 2])                       # 🙂 Warm, 🔥 Hot, 🔥 Hot
rw = torch.tensor([0.5, 6.0, 6.0])
nx = torch.tensor([2, 2, 0])                       # → 🔥 Hot, 🔥 Hot, 😴 Cold

tg = td_target(demo_critic, d, rw, nx)
print("targets:", tg.numpy().round(3))
assert abs(float(tg[2]) - 6.0) < 1e-6, "The day-2 step has no tomorrow: its target is just the reward."
assert not tg.requires_grad, "The target must be detached from the graph."
print("✅ the last one is exactly its reward (6.0): the campaign ends there, so nothing is added.")
print("critic loss on these three:", round(float(critic_loss(V_hat(demo_critic, d, st), tg)), 3))

### Does it actually learn? — freeze the actor and find out

Before we let the critic and the actor loose on each other, let's check the critic alone. We hold
`theta_pi` fixed, run campaigns, and train **only** the critic on the transitions they produce. Then
we compare what it believes against `V_PI` — the exact answer, which it has never seen and which
depends on a transition table it has no access to.

In [ ]:
#@title 🔧 Plumbing — collect a batch of transitions, and train a critic on a frozen policy  { display-mode: "form" }
# Two helpers you do not need to read. `collect` runs a batch of campaigns and flattens them into one
# pile of (day, state, reward, next state, log-prob) tensors; `train_critic_only` is Task 4's loss in a
# loop, with the actor held still. Run the cell — `collect` comes back in the A2C loop later.

def collect(theta, batch_size):
    '''Run a batch of campaigns and flatten them into one pile of transitions.'''
    days, states, rewards, next_states, log_probs = [], [], [], [], []
    for _ in range(batch_size):
        s, a, r, s_next, lp = sample_episode(theta)
        days        += list(range(N_DAYS))       # which morning each step happened on
        states      += s
        rewards     += r
        next_states += s_next
        log_probs   += lp
    return (torch.tensor(days), torch.tensor(states), torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states), torch.stack(log_probs))

def train_critic_only(theta, n_iters=600, batch_size=32, lr=0.05, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    critic = Critic()
    opt = torch.optim.Adam(critic.parameters(), lr=lr)
    for _ in range(n_iters):
        days, states, rewards, next_states, _ = collect(theta, batch_size)
        loss = critic_loss(V_hat(critic, days, states),
                           td_target(critic, days, rewards, next_states))
        opt.zero_grad(); loss.backward(); opt.step()
    return critic

print("Training a critic on a frozen policy…")
frozen_critic = train_critic_only(theta_pi)

with torch.no_grad():
    V_learned = np.array([[float(V_hat(frozen_critic, [t], [s])) for s in range(len(ENGAGE))]
                          for t in range(N_DAYS)])
ac_viz.critic_vs_truth(V_learned, V_PI)

> 👀 **Look at which cell it gets wrong.** Every entry is within a few rappen except *(day 0,
> 🔥 Hot)* — and no learner has ever been Hot on day 0, because nobody enters the campaign there. The
> critic is not bad at that cell; it has simply never been shown one, and is guessing by analogy with
> the Hot learners it *did* see. That is the honest failure mode of every learned value function:
> **it knows what your policy has bothered to visit.**

**It rebuilt the value function from nothing but sampled days.** No transition table, no
returns, no completed campaigns — a stack of `(s, a, r, s′)` tuples and a squared error. That is the
critic.

Two properties worth carrying:
- it **accumulates**. The batch-average baseline was recomputed from scratch and thrown away every
  iteration; this network keeps everything it has learned.
- it **generalises**. Ours has six inputs so there is nothing to generalise *to* — but replace the
  one-hot with a customer embedding and the same network gives you a value for a customer it has
  never seen. A per-state batch average cannot do that, and that is why this scales and that did not.

---
# Part 5 — Actor and critic, together

We now have both halves, and it is worth stating them side by side before we merge them, because
they are genuinely two different machine-learning problems sharing one stream of data.

In [ ]:
ac_viz.actor_critic_diagram()

### This loop should look familiar

Two boxes that take turns: one **acts**, the other **evaluates**, and the evaluation tells the first
one how to act better. That is **policy iteration** from the first lecture — with the exact
evaluation step replaced by regression on samples, and the greedy improvement step replaced by a
small gradient nudge.

In [ ]:
ac_viz.policy_iteration_cycle()

### The problem you may already have spotted

In [ ]:
ac_viz.moving_target_warning()

### 🧠 Quick check — one batch, one update

In [ ]:
ac_viz.mc_quiz("policy_iteration")

---
# Part 6 — A2C: the algorithm

One last ingredient, and it is there for a failure mode you can cause yourself in about four clicks.

## 6.1 · Why we pay the actor to stay undecided

The actor's update pushes probability towards whatever had a positive advantage. Early in training
those advantages come from a critic that is **still wrong** — its predictions are barely better than
noise. A couple of unlucky batches in a row and one action's probability goes to 99%.

And that is a trap door, not a setback: a policy only ever collects evidence about actions it
actually **takes**. An action pushed to 0.3% is never sampled again, so it can never produce the
campaign that would have proved it was the right move all along.

The fix is to make certainty cost something. **Entropy** measures how undecided a distribution is:

$$H(\pi(\cdot\mid s)) \;=\; -\sum_a \pi(a\mid s)\,\log \pi(a\mid s)$$

Maximal (`log 3 ≈ 1.10`) when all three moves are equally likely, zero when one of them has all the
probability. We **add** `c·H` to what we maximise, so the actor is paid a small amount for keeping its
options open — and only overrules that payment when the advantages keep insisting.

In [ ]:
ac_viz.entropy_playground()

In code that is one line, and we hand it to you — it is the only quantity in this notebook
that is about the policy alone: no reward, no critic, no world in it.

In [ ]:
def policy_entropy(theta, states):
    '''Average H(pi(.|s)) over a batch of visited states.'''
    p = torch.softmax(theta[states], dim=-1)          # one row of 3 probabilities per state
    return -(p * torch.log(p)).sum(dim=-1).mean()

visited = torch.tensor([0, 1, 2, 1, 1])               # any handful of visited states will do
print("H(uniform policy)         =",
      round(float(policy_entropy(torch.zeros(len(ENGAGE), len(ACTIONS)), visited)), 4),
      "  <- log 3 =", round(float(np.log(3)), 4), ": maximally undecided")
print("H(almost-decided policy)  =",
      round(float(policy_entropy(torch.tensor([[5.0, -5.0, -5.0]] * len(ENGAGE)), visited)), 4),
      "  <- nearly 0: nothing left to explore")
print("H(the policy from Part 2) =", round(float(policy_entropy(theta_pi, visited)), 4))

### 🧠 Quick check — the entropy bonus

In [ ]:
ac_viz.mc_quiz("entropy")

## 6.2 · The whole algorithm

Every piece is now on the table. **A2C — Advantage Actor–Critic** — assembles them in the obvious
way: run a batch of campaigns, ask the critic for the advantages, build one loss out of three terms,
take one step on each network, throw the batch away.

In [ ]:
ac_viz.a2c_loop_diagram()

### The three terms of the loss, and why each has its sign

$$L \;=\;
\underbrace{-\,\delta \cdot \log \pi_\theta(a\mid s)}_{\text{actor: } \textbf{maximise } \delta\log\pi}
\;+\;
\underbrace{c_V \cdot \big(V_\phi(s) - \text{target}\big)^2}_{\text{critic: } \textbf{minimise } \text{the error}}
\;-\;
\underbrace{c_H \cdot H(\pi_\theta(\cdot\mid s))}_{\text{explorer: } \textbf{maximise } \text{entropy}}$$

Autograd minimises, so anything we want *maximised* enters with a minus sign. Two of those three
terms are not losses in the usual sense at all — only the middle one predicts something. The other
two exist purely so that `backward()` produces the gradient we derived.

> 🧷 **One detail that breaks everything if you get it wrong.** The advantage `δ` in front of
> `log π` is the critic's **verdict on a move already made** — a number handed over, like a mark on a
> report. The actor's job is to change its *behaviour* in response to that mark. But `δ` was computed
> from the critic, so torch remembers the trail back to it, and left alone the optimiser will happily
> take the cheaper route: **improve the loss by shrinking the mark instead of by changing the
> behaviour** — an employee who improves their review by rewriting the review.
>
> `.detach()` is how we forbid that. It keeps the *value* of `δ` and cuts the trail leading back to
> the critic, so the number becomes a plain constant that nothing can be blamed for. The critic still
> learns — from its own regression term, on its own evidence — just not from the actor's wish that its
> verdict were kinder.

### 🎯 Task 5 — write the A2C iteration

Three blanks, all of them assembly of things you have already written. The scaffolding does the
collecting and the optimizer bookkeeping.

In [ ]:
def train_a2c(n_iters=300, batch_size=24, lr_actor=0.05, lr_critic=0.05,
              ent_coef=0.01, value_coef=0.5, gamma=GAMMA, seed=0, log_every=60):
    torch.manual_seed(seed); np.random.seed(seed)
    theta  = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)   # the ACTOR
    critic = Critic()                                                     # the CRITIC
    opt_actor  = torch.optim.Adam([theta], lr=lr_actor)
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr_critic)
    history, critic_error = [], []

    for it in range(n_iters):
        # 1 · COLLECT — a batch of campaigns under the CURRENT actor, flattened into transitions
        days, states, rewards, next_states, log_probs = collect(theta, batch_size)

        # 2 · ASK THE CRITIC — what did it expect, and what does the day say instead?
        v_now  = V_hat(critic, days, states)
        target = td_target(critic, days, rewards, next_states, gamma)
        advantage = target - ???    # 🎯 the TD error δ. `target` is already a plain constant (td_target
                                    #    built it inside no_grad) — the other one is not, so cut its
                                    #    trail back to the critic by ending it with .detach()

        # 3 · THREE TERMS, ONE LOSS
        actor_loss = -(???).mean()  # 🎯 δ times the log-prob of the move we made, for every step
        entropy    = policy_entropy(theta, states)
        loss = actor_loss + value_coef * critic_loss(v_now, target) - ent_coef * ???   # 🎯 which term?

        # 4 · ONE STEP on each network
        opt_actor.zero_grad(); opt_critic.zero_grad()
        loss.backward()
        opt_actor.step(); opt_critic.step()

        # 5 · REPEAT — the batch is discarded; it came from an actor that no longer exists.
        #     (Everything below is scorekeeping we can only afford in a toy world.)
        V_exact, _ = exact_values(theta, gamma)
        history.append(float(np.dot(START_PROBS, V_exact[0])))
        with torch.no_grad():
            V_now_all = np.array([[float(V_hat(critic, [t], [s])) for s in range(len(ENGAGE))]
                                  for t in range(N_DAYS)])
        critic_error.append(float(np.abs(V_now_all - V_exact).mean()))
        if log_every and (it + 1) % log_every == 0:
            print(f"  iter {it+1:4d}   J(θ) = {history[-1]:+.3f}   critic error = "
                  f"{critic_error[-1]:.3f}   H(π) = {float(entropy):.3f}")

    return theta.detach(), critic, history, critic_error

print("Training the actor and the critic together…")
theta_a2c, critic_a2c, hist_a2c, err_a2c = train_a2c()
print(f"\nFinal J = {hist_a2c[-1]:+.3f}   ·   best possible sheet = {J_BEST:+.3f}")

### Did the actor learn?

In [ ]:
ac_viz.training_curve([("A2C", hist_a2c, "#4a5bd0")], j_start=J_UNIFORM, j_best=J_BEST)

### And did the critic keep up?

Remember what this curve is measuring: the gap between the critic's `V̂` and the **exact value of the
policy as it is right now** — a target that keeps moving underneath it, because the actor keeps
improving. A critic that never quite converges is not a bug here; it is the job description.

In [ ]:
ac_viz.critic_curve(err_a2c)

### 📄 The deliverable, again

In [ ]:
sheet_a2c = ac_viz.playbook(theta_a2c)     # reads pi(a|s) off the actor, and counts what it visits

print("The learned sheet scores J =", round(sheet_J(sheet_a2c), 3),
      " · the best possible sheet scores J =", round(J_BEST, 3))

**Same sheet as notebook 02** — *warm them up, then cash in at the top* — reached by an
algorithm that never once waited for a campaign to finish.

## 6.3 · So was the critic worth it?

Let's be careful here, because the honest answer is more interesting than the marketing one. Below is
the algorithm from notebook 02 — REINFORCE with the per-state batch average as its baseline — on the same
world, same batch size, same learning rate, plotted against today's.

In [ ]:
#@title 📊 Notebook 02's REINFORCE, re-run here for comparison  { display-mode: "form" }
# Nothing new in it: complete campaigns, return-to-go, and the per-engagement-level batch
# average as the baseline.

def train_reinforce(n_iters=300, batch_size=24, lr=0.05, gamma=GAMMA, seed=0):
    '''Notebook 02's algorithm, for reference: complete campaigns, batch-average baseline.'''
    torch.manual_seed(seed); np.random.seed(seed)
    theta = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    history = []
    for it in range(n_iters):
        batch = [sample_episode(theta) for _ in range(batch_size)]
        G = np.array([returns_to_go(r, gamma) for _, _, r, _, _ in batch])
        sums, counts = np.zeros(len(ENGAGE)), np.zeros(len(ENGAGE))
        for i, (states, _, _, _, _) in enumerate(batch):
            for t, s in enumerate(states):
                sums[s] += G[i, t]; counts[s] += 1
        baseline = np.where(counts > 0, sums / np.maximum(counts, 1), G.mean())

        loss = torch.zeros(())
        for i, (states, _, _, _, log_probs) in enumerate(batch):
            for t in range(N_DAYS):
                loss = loss - (G[i, t] - baseline[states[t]]) * log_probs[t]
        opt.zero_grad(); (loss / batch_size).backward(); opt.step()
        history.append(exact_J(theta, gamma))
    return theta.detach(), history

print("Re-running notebook 02's REINFORCE for comparison…")
_, hist_pg = train_reinforce()

ac_viz.training_curve([("A2C — learned critic, one-step advantages", hist_a2c, "#4a5bd0"),
                       ("REINFORCE — batch-average baseline (notebook 02)", hist_pg, "#dd8452")],
                      j_start=J_UNIFORM, j_best=J_BEST,
                      title="Same world, same budget — is the critic worth the trouble?")

### 🔬 Read that plot honestly

**On this problem, the critic does not win — and the reason is the lesson.**

REINFORCE gets going faster — it passes `J = 4` after about **40** batches, where A2C needs about
**60**. Its baseline is the average return-to-go per engagement level, computed fresh from 24
campaigns; with **three states**, that average is an excellent estimate from the very first batch and
costs nothing to obtain. A2C, meanwhile, spends its opening iterations with a critic that still says
roughly zero everywhere — so its advantages are close to raw one-step rewards, a genuinely worse
signal. Both end at the same optimum, because both are aiming at the same gradient.

So what did we buy?

1. **Updates that don't wait for an ending.** Every one of A2C's updates was computable the morning
   after the move. Nothing in it needed the campaign to close. Point a continuing process at
   REINFORCE — a recommender, a trading desk, a chat session — and there is no episode to wait for.
2. **A value estimate that persists and generalises.** The batch average is recomputed and binned 300
   times. The critic is a function: it accumulates, and it answers for situations it has not seen. The
   moment "the state" stops being one of three labels — a customer embedding, a game frame, a
   conversation — the per-state average is not merely worse, it is **undefined**, and the critic is
   the only option left.
3. **The variance win arrives with the horizon.** Our campaign is three days, so a full return
   survives only three rolls of the dice. At a thousand steps, the Monte-Carlo return is hopeless and
   the one-step estimate is barely affected.

> 🧭 **The takeaway for a manager, not a coder:** a critic is infrastructure. It costs you a slow
> start and an extra thing to tune, and it buys you an algorithm whose shape does not fall apart when
> the problem gets big. On a three-state toy that is a bad trade. It is why nobody runs plain
> REINFORCE on anything real.

### 🧠 Quick check — A2C

In [ ]:
ac_viz.true_false_quiz("a2c")

---
# 🎓 Wrap-up

| The idea | The formula | In one line |
|---|---|---|
| **Q-function** `Q^π(s,a)` | `𝔼[G_t \| s,a]` | do this move, then carry on as usual — what does it pay? |
| **Value function** `V^π(s)` | `Σ_a π(a\|s) Q^π(s,a)` | what is this situation worth to me at all? |
| **Advantage** `A^π(s,a)` | `Q^π(s,a) − V^π(s)` | better or worse than my own habit, here? |
| **The one-step identity** | `Q = r + γV(s′)` | why the critic only ever learns `V` |
| **TD error** `δ` | `r + γV(s′) − V(s)` | a one-day estimate of the advantage |
| **TD learning** | `V ← V + αδ` | learn a guess from a slightly better guess |
| **The actor** | `−δ·log π(a\|s)` | push up whatever beat the yardstick |
| **The critic** | `(V(s) − target)²` | plain regression on sampled days |
| **Entropy bonus** | `+ c·H(π)` | make certainty cost something, so it is earned |
| **A2C** | collect → δ → one loss → one step → discard | the whole thing |

**The four things worth carrying out of here**

1. **Everything we did was renaming.** `G_t` was already a sample of `Q`; the baseline was already
   trying to be `V`. Nothing new was invented — we just wrote down what those two terms *meant*, and
   the algorithm reorganised itself around the answer.
2. **Naming them removed the episode.** Once the weight on an action is `r + γV(s′) − V(s)`, learning
   is a per-step operation. That is what makes RL usable on processes that never end.
3. **When you don't know a function, learn it — and pay the price knowingly.** The critic converts
   variance into bias. Early on its advantages are wrong, which is exactly when the entropy bonus is
   keeping your options open.
4. **Everything carries a `π` superscript.** `V^π`, `Q^π`, `A^π` — all of them describe *the current
   policy*. Move the policy and they go stale, which is why one batch buys exactly one update. That
   single constraint is the most expensive thing in this notebook.

### Where this goes next
- **Notebook 04 — GAE & PPO:** a dial between the one-step and full-return advantage (**GAE**), and a
  way to safely take *several* updates from one batch instead of throwing it away (**PPO**). Both are
  direct attacks on the two costs we just accepted.
- **Notebook 05 — RL for LLMs:** the same actor and the same critic, where the "campaign" is a
  generated answer, the "moves" are tokens, and the reward comes from a preference model.

---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **V, Q, advantage, the one-step identity, TD
learning, the actor, the critic, entropy, on-policy.** The rules: **3 lives**, **10 seconds** per
question, and a wrong answer *or* a timeout costs a life. Reach **10 correct** to pass. Good luck. 🍀

In [ ]:
ac_viz.flash_quiz()